In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# ─── CONFIGURE PATHS ─────────────────────────────────────────────────────────
# Change this to your server-mounted path or local copy
EXPERIMENTS_ROOT = Path(r"experiments")  # or Path("/home/dani00003/mCREAM/experiments")

DATASETS = ["CelebA", "Complete_Concept_FMNIST"]
METHODS = ["intersection", "majority", "union", "edge", "graph"]
M_VALUES = [5, 10]
DISAGREEMENT_LEVELS = ["low", "medium", "high", "structured_bias"]

print(f"Looking for results in: {EXPERIMENTS_ROOT.resolve()}")
print(f"Exists: {EXPERIMENTS_ROOT.exists()}")

## 1. Load All Results CSVs

In [ ]:
def load_all_results(root: Path) -> pd.DataFrame:
    """Scan experiment directories and load all per-seed result CSVs."""
    all_rows = []
    
    for dataset in DATASETS:
        mcream_dir = root / dataset / "train_cbm" / "mCREAM"
        if not mcream_dir.exists():
            print(f"  [SKIP] {mcream_dir} not found")
            continue
        
        for exp_dir in sorted(mcream_dir.iterdir()):
            if not exp_dir.is_dir():
                continue
            exp_name = exp_dir.name  # e.g., edge_M5_medium
            
            # Parse experiment name
            parts = exp_name.split("_")
            # Find method, M, level
            method = parts[0]  # edge, graph, intersection, majority, union
            m_val = None
            level = None
            for i, p in enumerate(parts):
                if p.startswith("M") and p[1:].isdigit():
                    m_val = int(p[1:])
                    level = "_".join(parts[i+1:])
                    break
            
            # Look for per-seed results
            for seed_dir in sorted(exp_dir.iterdir()):
                if not seed_dir.is_dir() or not seed_dir.name.startswith("seed_"):
                    continue
                seed = int(seed_dir.name.split("_")[1])
                
                # Find the CSV inside lightning_logs/version_X/
                for version_dir in sorted(seed_dir.glob("lightning_logs/version_*")):
                    csv_files = list(version_dir.glob("*.csv"))
                    # Main results CSV (not intervention_results)
                    main_csvs = [f for f in csv_files 
                                 if f.name != "intervention_results.csv"
                                 and "perc_" not in f.name
                                 and "seed" not in f.name]
                    if main_csvs:
                        for csv_f in main_csvs:
                            try:
                                df = pd.read_csv(csv_f)
                                df["dataset"] = dataset
                                df["method"] = method
                                df["M"] = m_val
                                df["disagreement_level"] = level
                                df["seed"] = seed
                                df["experiment_name"] = exp_name
                                df["source_file"] = str(csv_f)
                                all_rows.append(df)
                            except Exception as e:
                                print(f"    [ERR] {csv_f}: {e}")
    
    if not all_rows:
        print("No results found!")
        return pd.DataFrame()
    
    results = pd.concat(all_rows, ignore_index=True)
    print(f"\nLoaded {len(results)} result rows from {len(all_rows)} files")
    return results

results_df = load_all_results(EXPERIMENTS_ROOT)
if len(results_df) > 0:
    print(f"\nDatasets: {results_df['dataset'].unique()}")
    print(f"Methods: {results_df['method'].unique()}")
    print(f"M values: {results_df['M'].unique()}")
    print(f"Levels: {results_df['disagreement_level'].unique()}")
    results_df.head()

## 2. Load Intervention Results

In [ ]:
def load_all_interventions(root: Path) -> pd.DataFrame:
    """Load all intervention_results.csv files."""
    all_rows = []
    
    for dataset in DATASETS:
        mcream_dir = root / dataset / "train_cbm" / "mCREAM"
        if not mcream_dir.exists():
            continue
        
        for csv_f in mcream_dir.rglob("intervention_results.csv"):
            try:
                df = pd.read_csv(csv_f)
                # Extract metadata from path
                # .../mCREAM/edge_M5_medium/seed_42/lightning_logs/version_0/intervention_results.csv
                parts = csv_f.parts
                exp_name = None
                seed = None
                for i, p in enumerate(parts):
                    if p == "mCREAM" and i + 1 < len(parts):
                        exp_name = parts[i + 1]
                    if p.startswith("seed_"):
                        seed = int(p.split("_")[1])
                
                if exp_name:
                    name_parts = exp_name.split("_")
                    method = name_parts[0]
                    m_val = None
                    level = None
                    for i, p in enumerate(name_parts):
                        if p.startswith("M") and p[1:].isdigit():
                            m_val = int(p[1:])
                            level = "_".join(name_parts[i+1:])
                            break
                    
                    df["dataset"] = dataset
                    df["method"] = method
                    df["M"] = m_val
                    df["disagreement_level"] = level
                    df["seed"] = seed
                    df["experiment_name"] = exp_name
                    all_rows.append(df)
            except Exception as e:
                print(f"  [ERR] {csv_f}: {e}")
    
    if not all_rows:
        return pd.DataFrame()
    
    interventions_df = pd.concat(all_rows, ignore_index=True)
    print(f"Loaded {len(interventions_df)} intervention rows")
    return interventions_df

interventions_df = load_all_interventions(EXPERIMENTS_ROOT)
if len(interventions_df) > 0:
    print(interventions_df.head())

## 3. Summary Table: Task & Concept Accuracy (Mean ± Std across seeds)

In [ ]:
def make_summary_table(df, metric_cols, group_cols=["dataset", "method", "M", "disagreement_level"]):
    """Aggregate metrics across seeds: mean ± std."""
    available_cols = [c for c in metric_cols if c in df.columns]
    if not available_cols:
        print("No metric columns found!")
        return pd.DataFrame()
    
    grouped = df.groupby(group_cols)[available_cols]
    means = grouped.mean()
    stds = grouped.std()
    counts = grouped.count().iloc[:, 0].rename("n_seeds")
    
    # Format as mean ± std
    summary = pd.DataFrame(index=means.index)
    summary["n_seeds"] = counts
    for col in available_cols:
        summary[col] = means[col].map(lambda x: f"{x:.4f}") + " ± " + stds[col].map(lambda x: f"{x:.4f}")
    
    return summary

if len(results_df) > 0:
    # Core metrics
    core_metrics = ["test_task_accuracy", "test_concept_accuracy", 
                    "CCI", "PFI_concept_importance", "PFI_side_importance",
                    "concept_leakage", "c2y_baseline_accuracy"]
    
    summary = make_summary_table(results_df, core_metrics)
    print("=" * 80)
    print("CORE METRICS SUMMARY (mean ± std across seeds)")
    print("=" * 80)
    display(summary)

## 4. Graph Recovery Metrics

In [ ]:
if len(results_df) > 0:
    graph_metrics = ["u2c_f1", "u2c_precision", "u2c_recall",
                     "c2y_f1", "c2y_precision", "c2y_recall",
                     "u2c_num_edges_learned", "u2c_num_edges_gt",
                     "c2y_num_edges_learned", "c2y_num_edges_gt",
                     "learned_u2c_corruption_pct", "learned_c2y_corruption_pct"]
    
    graph_summary = make_summary_table(results_df, graph_metrics)
    print("=" * 80)
    print("GRAPH RECOVERY METRICS")
    print("=" * 80)
    display(graph_summary)

## 5. Task Accuracy Comparison Plot

In [ ]:
if len(results_df) > 0 and "test_task_accuracy" in results_df.columns:
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(7*len(DATASETS), 6), squeeze=False)
    
    for idx, dataset in enumerate(DATASETS):
        ax = axes[0, idx]
        subset = results_df[results_df["dataset"] == dataset].copy()
        if subset.empty:
            continue
        
        # Create label: method_M
        subset["config"] = subset["method"] + "_M" + subset["M"].astype(str)
        
        order = ["intersection_M5", "majority_M5", "union_M5", "edge_M5", "graph_M5",
                 "intersection_M10", "majority_M10", "union_M10", "edge_M10", "graph_M10"]
        order = [o for o in order if o in subset["config"].values]
        
        sns.boxplot(data=subset, x="config", y="test_task_accuracy", 
                    hue="disagreement_level", ax=ax, order=order)
        ax.set_title(f"Task Accuracy — {dataset}")
        ax.set_xlabel("")
        ax.tick_params(axis='x', rotation=45)
        ax.legend(title="Disagreement", loc="lower right")
    
    plt.tight_layout()
    plt.savefig("task_accuracy_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

## 6. Intervention Curves

In [ ]:
if len(interventions_df) > 0:
    # Plot intervention curves: accuracy vs num_interventions, grouped by method
    for dataset in DATASETS:
        for level in interventions_df["disagreement_level"].unique():
            subset = interventions_df[
                (interventions_df["dataset"] == dataset) & 
                (interventions_df["disagreement_level"] == level)
            ]
            if subset.empty:
                continue
            
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            
            for ax, mode in zip(axes, ["simple", "propagating"]):
                mode_data = subset[subset["mode"] == mode]
                if mode_data.empty:
                    ax.set_title(f"{mode} — no data")
                    continue
                
                # Mean across seeds per method+M+num_interventions
                agg = mode_data.groupby(["method", "M", "num_interventions"])["test_task_accuracy"].agg(["mean", "std"]).reset_index()
                
                for (method, m_val), grp in agg.groupby(["method", "M"]):
                    label = f"{method}_M{m_val}"
                    ax.errorbar(grp["num_interventions"], grp["mean"], 
                               yerr=grp["std"], label=label, marker='o', markersize=3, capsize=2)
                
                ax.set_xlabel("# Interventions")
                ax.set_ylabel("Task Accuracy")
                ax.set_title(f"{mode.capitalize()} Interventions — {dataset} ({level})")
                ax.legend(fontsize=8)
            
            plt.tight_layout()
            plt.savefig(f"interventions_{dataset}_{level}.png", dpi=150, bbox_inches="tight")
            plt.show()

## 7. Graph Recovery: F1 Scores (Edge vs Graph vs Baselines)

In [ ]:
if len(results_df) > 0 and "u2c_f1" in results_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for ax, graph_type in zip(axes, ["u2c", "c2y"]):
        col = f"{graph_type}_f1"
        if col not in results_df.columns:
            continue
        
        subset = results_df.dropna(subset=[col]).copy()
        subset["config"] = subset["method"] + "_M" + subset["M"].astype(str)
        
        sns.barplot(data=subset, x="disagreement_level", y=col, 
                   hue="config", ax=ax, errorbar="sd")
        ax.set_title(f"{graph_type.upper()} Graph Recovery (F1)")
        ax.set_ylim(0, 1)
        ax.legend(fontsize=7, ncol=2)
    
    plt.tight_layout()
    plt.savefig("graph_recovery_f1.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. CCI and PFI: Concept Reliance

In [ ]:
if len(results_df) > 0:
    cci_pfi_cols = ["CCI", "PFI_concept_importance", "PFI_side_importance"]
    available = [c for c in cci_pfi_cols if c in results_df.columns]
    
    if available:
        fig, axes = plt.subplots(1, len(available), figsize=(5*len(available), 5))
        if len(available) == 1:
            axes = [axes]
        
        for ax, col in zip(axes, available):
            subset = results_df.dropna(subset=[col]).copy()
            if subset.empty:
                continue
            subset["config"] = subset["method"] + "_M" + subset["M"].astype(str)
            
            sns.boxplot(data=subset, x="config", y=col, hue="dataset", ax=ax)
            ax.set_title(col)
            ax.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.savefig("cci_pfi_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()

## 9. Concept Leakage Analysis

In [ ]:
if len(results_df) > 0 and "concept_leakage" in results_df.columns:
    leakage_df = results_df.dropna(subset=["concept_leakage"]).copy()
    
    if not leakage_df.empty:
        leakage_df["config"] = leakage_df["method"] + "_M" + leakage_df["M"].astype(str)
        
        fig, ax = plt.subplots(figsize=(12, 5))
        sns.barplot(data=leakage_df, x="config", y="concept_leakage", 
                   hue="disagreement_level", ax=ax, errorbar="sd")
        ax.set_title("Concept Leakage: Λ = max(ACC_model - ACC_optimal, 0)")
        ax.set_ylabel("Leakage (lower = better)")
        ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label="No leakage")
        ax.tick_params(axis='x', rotation=45)
        ax.legend()
        plt.tight_layout()
        plt.savefig("concept_leakage.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("No leakage data available.")

## 10. Learned Graph Visualization (Example)

In [ ]:
def load_and_plot_learned_graph(exp_path: Path, title=""):
    """Load and visualize a learned soft graph vs ground truth with error highlighting."""
    soft_csv = exp_path / "learned_graphs" / "aggregated_dag_soft.csv"
    gt_csv = exp_path / "learned_graphs" / "ground_truth_dag.csv"
    
    if not soft_csv.exists():
        print(f"  No learned graph at {soft_csv}")
        return
    
    soft = pd.read_csv(soft_csv, index_col=0).astype(float)
    gt = pd.read_csv(gt_csv, index_col=0).astype(float) if gt_csv.exists() else None
    
    n_plots = 3 if gt is not None else 1
    fig, axes = plt.subplots(1, n_plots, figsize=(6*n_plots, 5))
    if n_plots == 1:
        axes = [axes]
    
    # Plot 1: Learned soft graph
    sns.heatmap(soft.values, ax=axes[0], vmin=0, vmax=1, 
                cmap="Blues", annot=True, fmt=".2f",
                xticklabels=soft.columns, yticklabels=soft.index)
    axes[0].set_title(f"Learned Soft Graph\n{title}")
    
    if gt is not None:
        # Plot 2: Ground truth
        sns.heatmap(gt.values, ax=axes[1], vmin=0, vmax=1,
                    cmap="Blues", annot=True, fmt=".0f",
                    xticklabels=gt.columns, yticklabels=gt.index)
        axes[1].set_title("Ground Truth")
        
        # Plot 3: Error map — shows where learned disagrees with GT
        # Green = correct, Red = error
        learned_binary = (soft.values > 0.5).astype(float)
        gt_binary = gt.values
        
        # 0 = correct (both match), 1 = false positive, -1 = false negative
        error_map = np.zeros_like(learned_binary)
        error_map[(learned_binary == 1) & (gt_binary == 0)] = 1    # FP (red)
        error_map[(learned_binary == 0) & (gt_binary == 1)] = -1   # FN (blue)
        # 0 = correct (white)
        
        sns.heatmap(error_map, ax=axes[2], vmin=-1, vmax=1,
                    cmap="RdBu_r", center=0, annot=True, fmt=".0f",
                    xticklabels=soft.columns, yticklabels=soft.index)
        axes[2].set_title("Errors\n(Red=FP spurious, Blue=FN missed, White=correct)")
    
    plt.tight_layout()
    plt.show()

# Example: load first available experiment's learned graph
if EXPERIMENTS_ROOT.exists():
    for dataset in DATASETS:
        mcream_dir = EXPERIMENTS_ROOT / dataset / "train_cbm" / "mCREAM"
        if not mcream_dir.exists():
            continue
        for exp_dir in sorted(mcream_dir.iterdir()):
            for seed_dir in sorted(exp_dir.glob("seed_*/lightning_logs/version_*")):
                if (seed_dir / "learned_graphs" / "aggregated_dag_soft.csv").exists():
                    load_and_plot_learned_graph(seed_dir, f"{exp_dir.name} / {seed_dir.parent.parent.name}")
                    break
            else:
                continue
            break  # Only show first example per dataset

## 11. Exogenous Correlation Analysis

In [ ]:
def load_and_plot_correlation(exp_path: Path, title=""):
    """Load and plot the exogenous correlation matrix."""
    corr_csv = exp_path / "correlation_analysis" / "exogenous_correlation_matrix.csv"
    abs_corr_csv = exp_path / "correlation_analysis" / "exogenous_abs_correlation_matrix.csv"
    
    if not corr_csv.exists():
        return
    
    corr = pd.read_csv(corr_csv, index_col=0)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr.values.astype(float), ax=ax, vmin=-1, vmax=1, 
                cmap="coolwarm", center=0,
                xticklabels=corr.columns, yticklabels=corr.index)
    ax.set_title(f"Exogenous Correlation\n{title}")
    plt.tight_layout()
    plt.show()

# Example
if EXPERIMENTS_ROOT.exists():
    for dataset in DATASETS:
        mcream_dir = EXPERIMENTS_ROOT / dataset / "train_cbm" / "mCREAM"
        if not mcream_dir.exists():
            continue
        for exp_dir in sorted(mcream_dir.iterdir()):
            for seed_dir in sorted(exp_dir.glob("seed_*/lightning_logs/version_*")):
                if (seed_dir / "correlation_analysis" / "exogenous_correlation_matrix.csv").exists():
                    load_and_plot_correlation(seed_dir, f"{exp_dir.name} / {seed_dir.parent.parent.name}")
                    break
            else:
                continue
            break

## 12. Expert Weights (Graph-level only)

In [ ]:
if len(results_df) > 0:
    # Expert weights columns: expert_weights_u2c, expert_weights_c2y
    weight_cols = [c for c in results_df.columns if "expert_weights" in c]
    if weight_cols:
        graph_only = results_df[results_df["method"] == "graph"].copy()
        if not graph_only.empty:
            print("Expert Weights (Graph-level aggregation):")
            print("="*60)
            for _, row in graph_only.iterrows():
                print(f"  {row['experiment_name']} (seed={row['seed']}):")
                for col in weight_cols:
                    if pd.notna(row.get(col)):
                        print(f"    {col}: {row[col]}")
                print()

## 13. Timing & Efficiency

In [ ]:
if len(results_df) > 0:
    timing_cols = ["train_val_time", "test_time", "num_trainable_parameters", 
                   "num_effective_parameters", "disconnected_weights", "peak_gpu_memory_mb"]
    available = [c for c in timing_cols if c in results_df.columns]
    
    if available:
        timing_summary = make_summary_table(results_df, available)
        print("TIMING & EFFICIENCY")
        print("="*80)
        display(timing_summary)

## 14. Export Final Summary Table (LaTeX-ready)

In [ ]:
if len(results_df) > 0:
    # Create a compact table for the paper
    key_metrics = ["test_task_accuracy", "test_concept_accuracy", "CCI", 
                   "u2c_f1", "c2y_f1", "concept_leakage"]
    available_key = [c for c in key_metrics if c in results_df.columns]
    
    # Aggregate: mean ± std as numeric
    group_cols = ["dataset", "method", "M", "disagreement_level"]
    agg = results_df.groupby(group_cols)[available_key].agg(["mean", "std"]).round(4)
    
    # Save to CSV
    agg.to_csv("mcream_summary_table.csv")
    print("Saved summary to mcream_summary_table.csv")
    
    # Display
    display(agg)

## 15. Quick Sanity Checks

In [ ]:
if len(results_df) > 0:
    print("SANITY CHECKS")
    print("=" * 60)
    
    # Check: edge/graph should beat baselines on graph recovery
    if "u2c_f1" in results_df.columns:
        for dataset in results_df["dataset"].unique():
            ds = results_df[results_df["dataset"] == dataset]
            for level in ds["disagreement_level"].unique():
                lv = ds[ds["disagreement_level"] == level]
                baselines_f1 = lv[lv["method"].isin(["intersection", "majority", "union"])]["u2c_f1"].mean()
                learnable_f1 = lv[lv["method"].isin(["edge", "graph"])]["u2c_f1"].mean()
                status = "✓" if learnable_f1 >= baselines_f1 else "✗"
                print(f"  {status} {dataset}/{level}: learnable F1={learnable_f1:.3f} vs baselines F1={baselines_f1:.3f}")
    
    # Check: CCI should be > 0.5 (concepts dominate)
    if "CCI" in results_df.columns:
        mean_cci = results_df["CCI"].dropna().mean()
        print(f"\n  Mean CCI across all experiments: {mean_cci:.3f} {'✓ > 0.5' if mean_cci > 0.5 else '✗ < 0.5'}")
    
    # Check: concept leakage should be near 0
    if "concept_leakage" in results_df.columns:
        mean_leak = results_df["concept_leakage"].dropna().mean()
        print(f"  Mean leakage: {mean_leak:.4f} {'✓ low' if mean_leak < 0.05 else '⚠ check'}")
    
    # Missing results check
    print(f"\n  Total experiments: {len(results_df)}")
    expected = len(DATASETS) * len(METHODS) * len(M_VALUES) * len(DISAGREEMENT_LEVELS) * 5  # 5 seeds
    print(f"  Expected (full grid): {expected}")
    print(f"  Coverage: {len(results_df)/expected*100:.0f}%")